# 02 Tokenizers Training

HuggingFace Tokenizers - Training Examples
==========================================

Complete examples for training different types of tokenizers.

Installation:
    pip install tokenizers

Time: 30-45 minutes

## Setup

Import required libraries:

In [ ]:
import os
from pathlib import Path


# =============================================================================
# Example 1: Train BPE Tokenizer (GPT-2 Style)
# =============================================================================

## Train Bpe Tokenizer

Train a BPE tokenizer from scratch

In [ ]:
def train_bpe_tokenizer():
    """Train a BPE tokenizer from scratch"""
    print("\n" + "=" * 70)
    print("EXAMPLE 1: Train BPE Tokenizer (GPT-2 Style)")
    print("=" * 70)
    
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPreTokenizer
    from tokenizers.decoders import ByteLevel as ByteLevelDecoder
    from tokenizers.processors import ByteLevel as ByteLevelProcessor
    
    # 1. Initialize
    print("\n📦 Initializing BPE tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="<|endoftext|>"))
    
    # 2. Pre-tokenizer (GPT-2 style byte-level)
    tokenizer.pre_tokenizer = ByteLevelPreTokenizer(add_prefix_space=True)
    
    # 3. Decoder
    tokenizer.decoder = ByteLevelDecoder()
    
    # 4. Trainer configuration
    trainer = BpeTrainer(
        vocab_size=5000,
        min_frequency=2,
        special_tokens=["<|endoftext|>"],
        show_progress=True
    )
    
    # 5. Training data
    training_data = [
        "The quick brown fox jumps over the lazy dog.",
        "Machine learning is transforming the world of AI.",
        "Natural language processing helps computers understand text.",
        "Tokenizers split text into manageable pieces.",
        "Deep learning models require proper tokenization.",
    ] * 500  # Repeat for better vocab
    
    print("📚 Training tokenizer...")
    tokenizer.train_from_iterator(training_data, trainer=trainer)
    
    # 6. Test
    test_text = "The tokenizer learned to split words!"
    output = tokenizer.encode(test_text)
    
    print(f"\n✅ Training complete!")
    print(f"📊 Vocabulary size: {tokenizer.get_vocab_size()}")
    print(f"\n📝 Test: '{test_text}'")
    print(f"🔢 Tokens: {output.tokens}")
    print(f"🆔 IDs: {output.ids[:10]}... (showing first 10)")
    
    # 7. Save
    save_path = "tokenizers/bpe_gpt2_style.json"
    os.makedirs("tokenizers", exist_ok=True)
    tokenizer.save(save_path)
    print(f"\n💾 Saved to: {save_path}")
    
    return tokenizer


# =============================================================================
# Example 2: Train WordPiece Tokenizer (BERT Style)
# =============================================================================

## Train Wordpiece Tokenizer

Train a WordPiece tokenizer like BERT

In [ ]:
def train_wordpiece_tokenizer():
    """Train a WordPiece tokenizer like BERT"""
    print("\n" + "=" * 70)
    print("EXAMPLE 2: Train WordPiece Tokenizer (BERT Style)")
    print("=" * 70)
    
    from tokenizers import Tokenizer
    from tokenizers.models import WordPiece
    from tokenizers.trainers import WordPieceTrainer
    from tokenizers.normalizers import BertNormalizer
    from tokenizers.pre_tokenizers import Whitespace
    from tokenizers.processors import TemplateProcessing
    from tokenizers.decoders import WordPiece as WordPieceDecoder
    
    # 1. Initialize
    print("\n📦 Initializing WordPiece tokenizer...")
    tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
    
    # 2. Normalizer (BERT-style)
    tokenizer.normalizer = BertNormalizer(
        clean_text=True,
        handle_chinese_chars=True,
        strip_accents=True,
        lowercase=True
    )
    
    # 3. Pre-tokenizer
    tokenizer.pre_tokenizer = Whitespace()
    
    # 4. Trainer
    trainer = WordPieceTrainer(
        vocab_size=10000,
        min_frequency=2,
        special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
        continuing_subword_prefix="##",
        show_progress=True
    )
    
    # 5. Training data (simulate diverse corpus)
    training_data = [
        "The quick brown fox jumps over the lazy dog.",
        "BERT uses WordPiece tokenization.",
        "Natural language understanding is improving rapidly.",
        "Machine learning models need quality data.",
        "Tokenization is the first step in NLP.",
        "Understanding context is crucial for language models.",
    ] * 500
    
    print("📚 Training WordPiece tokenizer...")
    tokenizer.train_from_iterator(training_data, trainer=trainer)
    
    # 6. Post-processor (BERT-style)
    tokenizer.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]",
        pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[
            ("[CLS]", tokenizer.token_to_id("[CLS]")),
            ("[SEP]", tokenizer.token_to_id("[SEP]")),
        ],
    )
    
    # 7. Decoder
    tokenizer.decoder = WordPieceDecoder(prefix="##")
    
    # 8. Test
    print(f"\n✅ Training complete!")
    print(f"📊 Vocabulary size: {tokenizer.get_vocab_size()}")
    
    test_text = "Understanding tokenization"
    output = tokenizer.encode(test_text)
    print(f"\n📝 Test: '{test_text}'")
    print(f"🔢 Tokens: {output.tokens}")
    decoded = tokenizer.decode(output.ids)
    print(f"🔄 Decoded: '{decoded}'")
    
    # 9. Save
    save_path = "tokenizers/wordpiece_bert_style.json"
    tokenizer.save(save_path)
    print(f"\n💾 Saved to: {save_path}")
    
    return tokenizer


# =============================================================================
# Example 3: Train Unigram Tokenizer (SentencePiece Style)
# =============================================================================

## Train Unigram Tokenizer

Train a Unigram tokenizer for multilingual use

In [ ]:
def train_unigram_tokenizer():
    """Train a Unigram tokenizer for multilingual use"""
    print("\n" + "=" * 70)
    print("EXAMPLE 3: Train Unigram Tokenizer (SentencePiece Style)")
    print("=" * 70)
    
    from tokenizers import Tokenizer
    from tokenizers.models import Unigram
    from tokenizers.trainers import UnigramTrainer
    from tokenizers.normalizers import NFKC
    from tokenizers.pre_tokenizers import Metaspace
    from tokenizers.decoders import Metaspace as MetaspaceDecoder
    
    # 1. Initialize
    print("\n📦 Initializing Unigram tokenizer...")
    tokenizer = Tokenizer(Unigram())
    
    # 2. Normalizer
    tokenizer.normalizer = NFKC()
    
    # 3. Pre-tokenizer (Metaspace for SentencePiece compatibility)
    tokenizer.pre_tokenizer = Metaspace()
    
    # 4. Decoder
    tokenizer.decoder = MetaspaceDecoder()
    
    # 5. Trainer
    trainer = UnigramTrainer(
        vocab_size=8000,
        special_tokens=["<unk>", "<s>", "</s>", "<pad>"],
        unk_token="<unk>",
        show_progress=True
    )
    
    # 6. Multilingual training data
    training_data = [
        # English
        "The quick brown fox jumps over the lazy dog.",
        "Machine learning is transforming the world.",
        # Spanish
        "El rápido zorro marrón salta sobre el perro perezoso.",
        "El aprendizaje automático está transformando el mundo.",
        # French
        "Le rapide renard brun saute par-dessus le chien paresseux.",
        "L'apprentissage automatique transforme le monde.",
        # German
        "Der schnelle braune Fuchs springt über den faulen Hund.",
        "Maschinelles Lernen verändert die Welt.",
    ] * 300
    
    print("📚 Training Unigram tokenizer...")
    tokenizer.train_from_iterator(training_data, trainer=trainer)
    
    # 7. Test on multiple languages
    print(f"\n✅ Training complete!")
    print(f"📊 Vocabulary size: {tokenizer.get_vocab_size()}")
    
    test_cases = [
        ("English", "Machine learning is amazing"),
        ("Spanish", "El aprendizaje es increíble"),
        ("French", "L'apprentissage est incroyable"),
    ]
    
    print("\n📝 Testing on multiple languages:")
    for lang, text in test_cases:
        output = tokenizer.encode(text)
        print(f"\n  {lang}: '{text}'")
        print(f"  Tokens: {output.tokens[:10]}... ({len(output.tokens)} total)")
    
    # 8. Save
    save_path = "tokenizers/unigram_multilingual.json"
    tokenizer.save(save_path)
    print(f"\n💾 Saved to: {save_path}")
    
    return tokenizer


# =============================================================================
# Example 4: Train Domain-Specific Tokenizer (Code)
# =============================================================================

## Train Code Tokenizer

Train a tokenizer optimized for code

In [ ]:
def train_code_tokenizer():
    """Train a tokenizer optimized for code"""
    print("\n" + "=" * 70)
    print("EXAMPLE 4: Train Code-Specific Tokenizer")
    print("=" * 70)
    
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace
    
    # 1. Initialize
    print("\n📦 Initializing code tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="<UNK>"))
    
    # 2. Pre-tokenizer (simple whitespace, preserve case)
    tokenizer.pre_tokenizer = Whitespace()
    
    # 3. Trainer with code-specific tokens
    trainer = BpeTrainer(
        vocab_size=15000,
        special_tokens=[
            "<UNK>", "<PAD>", "<BOS>", "<EOS>",
            "<INDENT>", "<DEDENT>", "<NEWLINE>",
            "<COMMENT>", "<STRING>", "<NUMBER>"
        ],
        show_progress=True
    )
    
    # 4. Code training data
    training_data = [
        # Python
        "def hello_world():\n    print('Hello, world!')\n    return True",
        "class MyClass:\n    def __init__(self):\n        self.value = 42",
        "for i in range(10):\n    print(i)",
        "import numpy as np\nimport pandas as pd",
        # JavaScript
        "function hello() {\n  console.log('Hello');\n  return true;\n}",
        "const myArray = [1, 2, 3, 4, 5];",
        "for (let i = 0; i < 10; i++) {\n  console.log(i);\n}",
        # General patterns
        "if (condition) { doSomething(); }",
        "var x = 10;\nvar y = 20;\nvar sum = x + y;",
    ] * 300
    
    print("📚 Training on code samples...")
    tokenizer.train_from_iterator(training_data, trainer=trainer)
    
    # 5. Test on code snippets
    print(f"\n✅ Training complete!")
    print(f"📊 Vocabulary size: {tokenizer.get_vocab_size()}")
    
    code_examples = [
        ("Python", "def factorial(n):\n    return 1 if n <= 1 else n * factorial(n-1)"),
        ("JavaScript", "const sum = (a, b) => a + b;"),
        ("General", "for i in range(10): print(i)")
    ]
    
    print("\n📝 Testing on code:")
    for lang, code in code_examples:
        output = tokenizer.encode(code)
        print(f"\n  {lang}:")
        print(f"  Code: {code[:50]}...")
        print(f"  Tokens: {output.tokens[:15]}...")
        print(f"  Total: {len(output.tokens)} tokens")
    
    # 6. Save
    save_path = "tokenizers/code_tokenizer.json"
    tokenizer.save(save_path)
    print(f"\n💾 Saved to: {save_path}")
    
    return tokenizer


# =============================================================================
# Example 5: Train with Real Files
# =============================================================================

## Train From Files

Train tokenizer from actual text files

In [ ]:
def train_from_files():
    """Train tokenizer from actual text files"""
    print("\n" + "=" * 70)
    print("EXAMPLE 5: Train from Files")
    print("=" * 70)
    
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace
    
    # 1. Create sample training files
    print("\n📝 Creating sample training files...")
    os.makedirs("training_data", exist_ok=True)
    
    # Create 3 sample files
    file_contents = [
        ("train.txt", "The quick brown fox jumps over the lazy dog.\n" * 100),
        ("valid.txt", "Machine learning is transforming AI.\n" * 100),
        ("test.txt", "Natural language processing is amazing.\n" * 100),
    ]
    
    files = []
    for filename, content in file_contents:
        filepath = f"training_data/{filename}"
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
        files.append(filepath)
        print(f"  Created: {filepath}")
    
    # 2. Initialize tokenizer
    print("\n📦 Initializing tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    
    # 3. Trainer
    trainer = BpeTrainer(
        vocab_size=5000,
        special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
        show_progress=True
    )
    
    # 4. Train from files
    print("\n📚 Training from files...")
    tokenizer.train(files, trainer)
    
    print(f"\n✅ Training complete!")
    print(f"📊 Vocabulary size: {tokenizer.get_vocab_size()}")
    
    # 5. Test
    test_text = "The fox is learning machine learning"
    output = tokenizer.encode(test_text)
    print(f"\n📝 Test: '{test_text}'")
    print(f"🔢 Tokens: {output.tokens}")
    
    # 6. Save
    save_path = "tokenizers/file_trained.json"
    tokenizer.save(save_path)
    print(f"\n💾 Saved to: {save_path}")
    
    return tokenizer


# =============================================================================
# Example 6: Compare Different Tokenizers
# =============================================================================

## Compare Tokenizers

Compare tokenization results across different models

In [ ]:
def compare_tokenizers():
    """Compare tokenization results across different models"""
    print("\n" + "=" * 70)
    print("EXAMPLE 6: Compare Different Tokenizers")
    print("=" * 70)
    
    from tokenizers import Tokenizer
    
    # Load previously trained tokenizers
    tokenizer_paths = {
        "BPE (GPT-2)": "tokenizers/bpe_gpt2_style.json",
        "WordPiece (BERT)": "tokenizers/wordpiece_bert_style.json",
        "Unigram": "tokenizers/unigram_multilingual.json",
        "Code": "tokenizers/code_tokenizer.json",
    }
    
    tokenizers = {}
    for name, path in tokenizer_paths.items():
        if os.path.exists(path):
            tokenizers[name] = Tokenizer.from_file(path)
    
    if not tokenizers:
        print("⚠️ No trained tokenizers found. Run other examples first.")
        return
    
    # Test sentences
    test_sentences = [
        "Machine learning is transforming the world.",
        "The quick brown fox jumps over the lazy dog.",
        "Understanding natural language processing.",
    ]
    
    print("\n📊 Comparing tokenization results:\n")
    
    for sentence in test_sentences:
        print(f"📝 Sentence: '{sentence}'")
        print("-" * 70)
        
        for name, tokenizer in tokenizers.items():
            output = tokenizer.encode(sentence)
            print(f"\n  {name}:")
            print(f"    Tokens: {output.tokens[:15]}")
            print(f"    Count: {len(output.tokens)} tokens")
            print(f"    IDs: {output.ids[:10]}...")
        
        print("\n")


# =============================================================================
# Example 7: Fine-tune Existing Tokenizer
# =============================================================================

## Finetune Tokenizer

Add domain-specific vocabulary to existing tokenizer

In [ ]:
def finetune_tokenizer():
    """Add domain-specific vocabulary to existing tokenizer"""
    print("\n" + "=" * 70)
    print("EXAMPLE 7: Fine-tune Existing Tokenizer")
    print("=" * 70)
    
    from tokenizers import Tokenizer
    
    # Load base tokenizer
    if not os.path.exists("tokenizers/wordpiece_bert_style.json"):
        print("⚠️ Base tokenizer not found. Train WordPiece tokenizer first.")
        return
    
    print("\n📥 Loading base tokenizer...")
    tokenizer = Tokenizer.from_file("tokenizers/wordpiece_bert_style.json")
    
    vocab_before = tokenizer.get_vocab_size()
    print(f"📊 Vocabulary size before: {vocab_before}")
    
    # Add domain-specific tokens
    print("\n➕ Adding domain-specific tokens...")
    new_tokens = [
        # Medical terms
        "covid19", "vaccine", "antibody", "pandemic",
        # Tech terms
        "tensorflow", "pytorch", "neural_network", "transformer",
        # Custom tokens
        "[ENTITY]", "[DATE]", "[NUMBER]", "[URL]"
    ]
    
    num_added = tokenizer.add_tokens(new_tokens)
    
    vocab_after = tokenizer.get_vocab_size()
    print(f"✅ Added {num_added} new tokens")
    print(f"📊 Vocabulary size after: {vocab_after}")
    
    # Test with domain-specific text
    test_text = "The covid19 vaccine uses tensorflow for neural_network prediction"
    output = tokenizer.encode(test_text)
    
    print(f"\n📝 Test: '{test_text}'")
    print(f"🔢 Tokens: {output.tokens}")
    
    # Save fine-tuned tokenizer
    save_path = "tokenizers/finetuned_tokenizer.json"
    tokenizer.save(save_path)
    print(f"\n💾 Saved to: {save_path}")


# =============================================================================
# Main Function
# =============================================================================

## Main

Run all training examples

In [ ]:
def main():
    """Run all training examples"""
    print("\n" + "=" * 70)
    print("HUGGINGFACE TOKENIZERS - TRAINING EXAMPLES")
    print("=" * 70)
    
    print("\nThis script demonstrates:")
    print("  1. Training BPE tokenizer (GPT-2 style)")
    print("  2. Training WordPiece tokenizer (BERT style)")
    print("  3. Training Unigram tokenizer (multilingual)")
    print("  4. Training domain-specific tokenizer (code)")
    print("  5. Training from files")
    print("  6. Comparing different tokenizers")
    print("  7. Fine-tuning existing tokenizers")
    
    try:
        # Train different types
        train_bpe_tokenizer()
        train_wordpiece_tokenizer()
        train_unigram_tokenizer()
        train_code_tokenizer()
        train_from_files()
        
        # Compare and fine-tune
        compare_tokenizers()
        finetune_tokenizer()
        
        print("\n" + "=" * 70)
        print("✅ ALL TRAINING EXAMPLES COMPLETED!")
        print("=" * 70)
        print(f"\n📁 Trained tokenizers saved in: ./tokenizers/")
        print("📚 Next steps:")
        print("  - Load and use these tokenizers in your projects")
        print("  - Experiment with different hyperparameters")
        print("  - Train on your own domain-specific data")
        print("\n")
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()